# Three-dimensional conservative RAD: local P4 and face P1

The oscillatory problem from Araya et al. (2024), section 5.2.1, uses K=0.1 I, beta=(1,0,0), and u=sin(6 pi x) sin(4 pi y) sin(2 pi z). This study preserves those coefficients and polynomial degrees, with explicitly different deterministic tetrahedral meshes. The classical P2 comparison has its own exact-solution refinement check.


The current main equations and their physical checks execute from source. The full acquisition and its field galleries run with `--study`; inspect `--study --plan` for locked commands and resource requirements. Exact original external comparisons require `--historical` and their attributed payloads.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/f9670cc4820faadfefed43a34e6d03ca2c17ac98c6f64e306727d549c3a137a3/41_rad3d-companion.zip"
COMPANION_SHA256 = "f9670cc4820faadfefed43a34e6d03ca2c17ac98c6f64e306727d549c3a137a3"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("transport/41_rad3d.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


In [ ]:
import hashlib
import json
import numpy as np
from IPython.display import Image, display
from pymhm import assemble, TetraMesh, TriangularSkeleton
from pymhm.fem.traces.triangle_3d import tetra_trace_coupling
from examples.formulations.scalar_3d import (
    Scalar3DDiscretization, scalar_equations_3d, scalar_problem_3d,
    define_scalar_3d, scalar_constraints_3d, recover_transport_3d,
    recover_scalar_3d, tetra_boundary_pairing,
)
from examples.formulations.transport_3d import define_transport_3d
from examples.formulations.original import solve_original
from examples.solve_rad3d import source, exact, gradient, physical_flux


In [ ]:
from inspect import getsource
print(getsource(scalar_equations_3d))
print(getsource(scalar_problem_3d))


## Small affine transport patch

The Robin multiplier is (-K grad(u)+beta u/2).n, whereas the physical conservative flux is -K grad(u)+beta u. Weak Dirichlet data prescribe face moments.


In [ ]:
def affine(x):
    """Return the manufactured scalar with nonhomogeneous Dirichlet data."""
    return 1 + x @ np.array([1.0, 2.0, 3.0])

mesh = TetraMesh.unit_cube(1)
data = Scalar3DDiscretization(
    mesh, TriangularSkeleton(mesh, degree=1),
    tetra_trace_coupling, tetra_boundary_pairing,
    degree=4, refinement=1,
)
problem = scalar_problem_3d(data, velocity=[1, 0, 0], source=1.0, dirichlet=affine)
system = assemble(problem)
solution = recover_scalar_3d(data, system, system.solve(), velocity=[1, 0, 0])
assert solution.l2_error(affine) < 1e-10
assert solution.h1_seminorm_error([1, 2, 3]) < 1e-9
print("Affine P4 / P1-face patch passed")


## Current oscillatory physical case through the public API

This execution uses the campaign's unit cube, coefficients, manufactured source,
P4 local fields, P1 triangular traces and local refinement 2. Its six macro
cells form the first deterministic campaign mesh; this is an analytical control,
not the article's unavailable historical connectivity. The selective constant
kernel declaration has no retained modes for this non-tangential velocity. A
second declaration retaining every constant gives the same physical coefficient
fields. The first system is also compared with independently assembled original
A/B/C/D rows, with the same spaces and quadrature.


In [ ]:
mesh = TetraMesh.unit_cube(1)
inputs = dict(degree=4, local_refinement=2, diffusion=.1,
              velocity=(1., 0., 0.), source=source, quadrature_order=8)
definition = define_transport_3d(mesh, coarse_space="kernel", **inputs)
system = assemble(definition.problem)
coefficients = system.solve(constraints=scalar_constraints_3d(definition, system))
physical = recover_transport_3d(definition, system, coefficients)
original = solve_original(system)
retained = define_transport_3d(mesh, coarse_space="constants", **inputs)
retained_system = assemble(retained.problem)
retained_coefficients = retained_system.solve()
for actual, direct, constant in zip(coefficients.fields, original.fields,
                                    retained_coefficients.fields, strict=True):
    np.testing.assert_allclose(actual, direct, atol=1e-12, rtol=1e-10)
    np.testing.assert_allclose(actual, constant, atol=1e-12, rtol=1e-10)
print({"macro_cells": len(mesh.cells), "trace_dofs": definition.problem.trace_size,
       "scalar_L2": physical.l2_error(exact, order=16),
       "gradient_L2": physical.h1_seminorm_error(gradient, order=16),
       "physical_flux_L2": physical.flux_l2_error(physical_flux, order=16)})


## Five-level analytical campaign

Scalar, gradient, and full physical flux norms use volume integration. The V norm includes L2/diameter squared, with diameter squared equal to three. Assembly and error quadrature use order eight; the finest errors are checked independently with order ten.


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    record = json.loads((root / "examples/results/rad3d.json").read_text())
    assert not record["source_changed_during_run"]
    assert [row["n"] for row in record["rows"]] == [1, 2, 3, 4, 5]
    for method in ("mhm", "classical"):
        rows = record["rows"]
        assert rows[-1][method]["V_error"] < rows[0][method]["V_error"]
        assert max(row[method]["backward_residual"] for row in rows) < 1e-10
        for norm in ("l2_error", "V_error", "physical_flux_l2_error"):
            np.testing.assert_allclose(rows[-1][method][norm], rows[-1]["quadrature_order10"][method][norm], rtol=2e-6, atol=1e-12)
        print(method, [(r["n"], r[method]["l2_error"], r[method]["V_error"]) for r in rows])
    last = record["rows"][-1]
    archive = root / "examples/results" / last["fields"]
    assert hashlib.sha256(archive.read_bytes()).hexdigest() == last["fields_sha256"]


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    for name in ("convergence", "scalar", "flux"):
        display(Image(filename=str(root / f"docs/figures/rad3d/{name}.png")))


The field panels keep individual one-sided tetrahedral values and show the actual
macro intersections on every map. The methods use shared physical limits and
centered difference limits. Native optional tests independently assemble P3/P4
Galerkin and SUPG matrices and loads in DOLFINx/UFL.
